In [1]:
%pip uninstall -y numpy
%pip install "numpy>=1.26.4,<2"

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Note: you may need to restart the kernel to use updated packages.
  Using cached numpy-1.26.4-cp310-cp310-macosx_10_9_x86_64.whl (20.6 MB)
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
You should consider upgrading via the '/usr/local/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install transformers
%pip install torch
%pip install -U sentence-transformers nltk

You should consider upgrading via the '/usr/local/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/usr/local/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.6/486.6 KB 3.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 4.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.1/25.1 MB 5.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 4.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 4.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.4/308.4 KB 3.0 MB/s eta 0:00:0000:0100:01
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
You should con

In [1]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("fill-mask", model="FacebookAI/roberta-base")

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cpu


In [2]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")
model = AutoModelForMaskedLM.from_pretrained("FacebookAI/roberta-base")

In [3]:
pipe("The capital of France is <mask>.")


[{'score': 0.9036266207695007,
  'token': 2201,
  'token_str': ' Paris',
  'sequence': 'The capital of France is Paris.'},
 {'score': 0.0802929550409317,
  'token': 12790,
  'token_str': ' Lyon',
  'sequence': 'The capital of France is Lyon.'},
 {'score': 0.0048033325001597404,
  'token': 16911,
  'token_str': ' Nice',
  'sequence': 'The capital of France is Nice.'},
 {'score': 0.0020991009660065174,
  'token': 8239,
  'token_str': ' Nancy',
  'sequence': 'The capital of France is Nancy.'},
 {'score': 0.001129904412664473,
  'token': 35767,
  'token_str': ' Napoleon',
  'sequence': 'The capital of France is Napoleon.'}]

In [4]:
# Imports
from sentence_transformers import SentenceTransformer, util
from nltk.tokenize import sent_tokenize
import nltk

In [5]:
nltk.download('punkt', download_dir='/tmp')

[nltk_data] Downloading package punkt to /tmp...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [6]:
# Load SentenceTransformer model (RoBERTa-based)
model = SentenceTransformer('sentence-transformers/roberta-base-nli-mean-tokens')

# Example reflective templates (expand as needed)
reflection_templates = [
    "I learned something important about myself.",
    "That experience changed me.",
    "I grew from that challenge.",
    "I gained a new perspective.",
    "I became more self-aware.",
    "It taught me a lesson.",
    "I realized something I hadn't before.",
    "This helped me understand myself better."
]

# Encode the reflection templates once
template_embeddings = model.encode(reflection_templates, convert_to_tensor=True)

# Paste your essay here (replace this with user input)
essay_text = """
I used to hate group projects because I preferred working alone. But after joining the robotics team, I realized that collaboration brings out the best in everyone. I learned how to trust my teammates and communicate better. This experience changed how I view leadership and teamwork.
"""

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [7]:
%pip install pysbd

import pysbd
seg = pysbd.Segmenter(language="en", clean=True)
sentences = seg.segment(essay_text)

sentence_embeddings = model.encode(sentences, convert_to_tensor=True)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


You should consider upgrading via the '/usr/local/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [8]:
# Check similarity to reflection templates
reflective_sentences = []
threshold = 0.6
total_score = 0

for i, sentence in enumerate(sentences):
    sim_scores = util.cos_sim(sentence_embeddings[i], template_embeddings)
    max_score = sim_scores.max().item()
    
    if max_score > threshold:
        reflective_sentences.append((sentence, round(max_score, 3)))
        total_score += 1

In [10]:
# Score as a percentage
reflection_score = round((total_score / len(sentences)) * 100, 2)

In [11]:
# Print results
print(f"\n Reflection Score: {reflection_score}/100")
print("\n Reflective Sentences Detected:")


 Reflection Score: 50.0/100

 Reflective Sentences Detected:


In [12]:
if reflective_sentences:
    for sent, sim in reflective_sentences:
        print(f"- \"{sent}\" (similarity: {sim})")
else:
    print("No strongly reflective sentences found.")

- "I learned how to trust my teammates and communicate better." (similarity: 0.829)
- "This experience changed how I view leadership and teamwork." (similarity: 0.811)
